# 06A - SIMCA Pure-Test Evaluation

This notebook evaluates reviewed SIMCA candidates on the held-out pure-test batch.

Protocol:

- train on pure target objects from batches 1, 2, and 3;
- project pure reference objects from batch 4 only;
- compute 2-way object and pixel metrics;
- apply 3-way thresholds fixed before the test, from notebook 04C validation calibration;
- compute image-level diagnostics;
- write guardrails proving that no pure-test data is used before this stage.

The notebook does not perform final model selection. Notebook 06B should consume these metrics.


## Inputs And Outputs

Main inputs:

- `results/03_pca_<RESULTS_TAG>/pca_selected_preprocessings.parquet`
- `results/04C_simca_concat_refit_<RESULTS_TAG>/candidate_panel.parquet`
- `results/04C_simca_concat_refit_<RESULTS_TAG>/validation_3way_selected_thresholds.parquet`
- `results/05_simca_validation_robustness_<RESULTS_TAG>/track_scoring_flags.parquet`

Main outputs:

- `pure_test_candidate_panel.parquet`
- `pure_test_2way_object_metrics.parquet`
- `pure_test_2way_pixel_metrics.parquet`
- `pure_test_3way_object_metrics.parquet`
- `pure_test_metrics_long.parquet`
- `pure_test_object_diagnostics_by_image.parquet`
- `pure_test_pixel_diagnostics_by_image.parquet`
- `pure_test_3way_object_diagnostics_by_image.parquet`
- `pure_test_guardrails.parquet`
- `pure_test_protocol.parquet`


In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from IPython.display import display

CURRENT_DIR = Path.cwd().resolve()
if (CURRENT_DIR / "src").exists():
    PROJECT_ROOT = CURRENT_DIR
elif (CURRENT_DIR.parent / "src").exists():
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    raise RuntimeError(
        "Could not find project root. Run this notebook from the project root or notebooks/."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("PROJECT_ROOT:", PROJECT_ROOT)

from src import experiment_config as expcfg
from src.io.database_h5 import load_nir_uco_h5
from src.utils import list_result_files
from src.spectra.band_selection import (
    select_wavelength_range_from_database,
    wavelength_selection_summary,
)
from src.workflows.simca_candidates import build_pca_preprocessing_configs_by_matrix_family
from src.workflows.simca import make_target_train_filters
from src.workflows.simca_pure_test import (
    DEFAULT_PURE_TEST_EVALUATION_STAGE,
    build_pure_test_guardrails,
    build_pure_test_projection_filters,
    build_pure_test_protocol,
    load_existing_pure_test_outputs,
    missing_existing_pure_test_paths,
    run_pure_test_refit_batches,
    save_pure_test_outputs,
    select_pure_test_candidate_panel,
    summarize_pure_test_outputs,
    validate_pure_test_guardrails,
    validate_pure_test_outputs,
)
from src.workflows.simca_tables import read_simca_table, write_simca_table


PROJECT_ROOT: C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts


## Configuration

Set `RUN_PURE_TEST_REFIT = False` and `USE_EXISTING_PURE_TEST_OUTPUTS = True` to reload already generated pure-test parquet files without refitting.

Use `MAX_PURE_TEST_CANDIDATES` or `MAX_PURE_TEST_CANDIDATES_PER_TRACK` only for debugging. A full pure-test run should keep both as `None`.


In [2]:
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DB_H5_PATH = PROJECT_ROOT / "HSI Data" / "processed" / "nir_uco_database.h5"

RESULTS_TAG = expcfg.DEFAULT_RESULTS_TAG
WAVELENGTH_MODE = expcfg.DEFAULT_WAVELENGTH_MODE

RESULTS_03_DIR = PROJECT_ROOT / "results" / f"03_pca_{RESULTS_TAG}"
RESULTS_04C_DIR = PROJECT_ROOT / "results" / f"04C_simca_concat_refit_{RESULTS_TAG}"
RESULTS_05_DIR = PROJECT_ROOT / "results" / f"05_simca_validation_robustness_{RESULTS_TAG}"
RESULTS_DIR = PROJECT_ROOT / "results" / f"06A_simca_pure_test_{RESULTS_TAG}"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

PCA_SELECTED_PREPROCESSINGS_PATH = RESULTS_03_DIR / "pca_selected_preprocessings.parquet"
CANDIDATE_PANEL_04C_PATH = RESULTS_04C_DIR / "candidate_panel.parquet"
VALIDATION_3WAY_SELECTED_THRESHOLDS_PATH = RESULTS_04C_DIR / "validation_3way_selected_thresholds.parquet"
TRACK_SCORING_FLAGS_05_PATH = RESULTS_05_DIR / "track_scoring_flags.parquet"

WAVELENGTH_CONFIG_PATH = RESULTS_DIR / "wavelength_config.parquet"
PREPROCESSING_SCOPE_PATH = RESULTS_DIR / "preprocessing_scope.parquet"

PURE_TEST_PATHS = {
    "candidate_panel": RESULTS_DIR / "pure_test_candidate_panel.parquet",
    "2way_object_metrics": RESULTS_DIR / "pure_test_2way_object_metrics.parquet",
    "2way_pixel_metrics": RESULTS_DIR / "pure_test_2way_pixel_metrics.parquet",
    "3way_object_metrics": RESULTS_DIR / "pure_test_3way_object_metrics.parquet",
    "metrics_long": RESULTS_DIR / "pure_test_metrics_long.parquet",
    "object_image_diagnostics": RESULTS_DIR / "pure_test_object_diagnostics_by_image.parquet",
    "pixel_image_diagnostics": RESULTS_DIR / "pure_test_pixel_diagnostics_by_image.parquet",
    "3way_object_image_diagnostics": RESULTS_DIR / "pure_test_3way_object_diagnostics_by_image.parquet",
    "pixel_errors_by_image": RESULTS_DIR / "pure_test_pixel_errors_by_image.parquet",
    "errors": RESULTS_DIR / "pure_test_errors.parquet",
    "guardrails": RESULTS_DIR / "pure_test_guardrails.parquet",
    "protocol": RESULTS_DIR / "pure_test_protocol.parquet",
    "batch_manifest": RESULTS_DIR / "pure_test_batch_manifest.parquet",
    "diagnostics": RESULTS_DIR / "pure_test_diagnostics.parquet",
    "objects": RESULTS_DIR / "pure_test_objects.parquet",
    "pixels": RESULTS_DIR / "pure_test_pixels.parquet",
    "3way_objects": RESULTS_DIR / "pure_test_3way_objects.parquet",
}

PURE_TEST_BATCH_DIR = RESULTS_DIR / "pure_test_batches"
PURE_TEST_BATCH_METRICS_DIR = PURE_TEST_BATCH_DIR / "metrics"
PURE_TEST_BATCH_OBJECTS_DIR = PURE_TEST_BATCH_DIR / "objects"
PURE_TEST_BATCH_PIXELS_DIR = PURE_TEST_BATCH_DIR / "pixels"
PURE_TEST_BATCH_3WAY_OBJECTS_DIR = PURE_TEST_BATCH_DIR / "objects_3way"
for _path in [
    PURE_TEST_BATCH_DIR,
    PURE_TEST_BATCH_METRICS_DIR,
    PURE_TEST_BATCH_OBJECTS_DIR,
    PURE_TEST_BATCH_PIXELS_DIR,
    PURE_TEST_BATCH_3WAY_OBJECTS_DIR,
]:
    _path.mkdir(parents=True, exist_ok=True)

TARGET_CLASS = expcfg.TARGET_CLASS
NON_TARGET_LABEL = expcfg.NON_TARGET_LABEL
REFERENCE_CLASSES = list(expcfg.REFERENCE_CLASSES)

PURE_TEST_TRAIN_BATCHES = list(expcfg.PURE_TEST_TRAIN_BATCHES)
PURE_TEST_BATCHES = list(expcfg.PURE_TEST_BATCHES)

TRAIN_FILTERS = make_target_train_filters(
    target_class=TARGET_CLASS,
    train_batches=PURE_TEST_TRAIN_BATCHES,
)
PURE_TEST_FILTERS = build_pure_test_projection_filters(
    reference_classes=REFERENCE_CLASSES,
    test_batches=PURE_TEST_BATCHES,
)

USE_WAVELENGTH_WINDOW = False
WINDOW_MIN_NM = 1225.0
WINDOW_MAX_NM = 1675.0

RUN_PURE_TEST_REFIT = True
USE_EXISTING_PURE_TEST_OUTPUTS = True
PURE_TEST_BATCH_SIZE = 50
MAX_PURE_TEST_CANDIDATES = None
MAX_PURE_TEST_CANDIDATES_PER_TRACK = None

RANDOM_STATE = expcfg.RANDOM_STATE
REPLACE_BALANCED_PIXELS = expcfg.REPLACE_BALANCED_PIXELS
CV_N_SPLITS = expcfg.CV_N_SPLITS
CV_GROUP_COL = expcfg.CV_GROUP_COL

# Memory-safe default profile.
# Metric and image-diagnostic tables are enough for notebook 06B final selection
# and notebook 07 mixture application. Detailed projection tables can be enabled
# later for a small set of final candidates only.
SAVE_BATCH_METRIC_TABLES = True
SAVE_BATCH_OBJECT_TABLES = False
SAVE_BATCH_PIXEL_TABLES = False
SAVE_BATCH_3WAY_OBJECT_TABLES = False
SAVE_COMBINED_OBJECT_TABLES = False
SAVE_COMBINED_PIXEL_TABLES = False
SAVE_COMBINED_3WAY_OBJECT_TABLES = False

EVALUATION_STAGE = DEFAULT_PURE_TEST_EVALUATION_STAGE

print("Input 04C dir:", RESULTS_04C_DIR)
print("Input 05 dir:", RESULTS_05_DIR)
print("Output dir:", RESULTS_DIR)
print("PURE_TEST_TRAIN_BATCHES:", PURE_TEST_TRAIN_BATCHES)
print("PURE_TEST_BATCHES:", PURE_TEST_BATCHES)
print("RUN_PURE_TEST_REFIT:", RUN_PURE_TEST_REFIT)
print("USE_EXISTING_PURE_TEST_OUTPUTS:", USE_EXISTING_PURE_TEST_OUTPUTS)
print("PURE_TEST_BATCH_SIZE:", PURE_TEST_BATCH_SIZE)
print("MAX_PURE_TEST_CANDIDATES:", MAX_PURE_TEST_CANDIDATES)
print("MAX_PURE_TEST_CANDIDATES_PER_TRACK:", MAX_PURE_TEST_CANDIDATES_PER_TRACK)


Input 04C dir: C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\04C_simca_concat_refit_non_noisy_all
Input 05 dir: C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\05_simca_validation_robustness_non_noisy_all
Output dir: C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\06A_simca_pure_test_non_noisy_all
PURE_TEST_TRAIN_BATCHES: [1, 2, 3]
PURE_TEST_BATCHES: [4]
RUN_PURE_TEST_REFIT: True
USE_EXISTING_PURE_TEST_OUTPUTS: True
PURE_TEST_BATCH_SIZE: 50
MAX_PURE_TEST_CANDIDATES: None
MAX_PURE_TEST_CANDIDATES_PER_TRACK: None


## Load Inputs And Guard Pure-Test Protocol

The pure-test batch is not used to select preprocessings, rules, thresholds, or models. The 3-way thresholds are loaded from 04C validation outputs and treated as fixed.


In [3]:
required_paths = [
    DB_H5_PATH,
    PCA_SELECTED_PREPROCESSINGS_PATH,
    CANDIDATE_PANEL_04C_PATH,
    VALIDATION_3WAY_SELECTED_THRESHOLDS_PATH,
    TRACK_SCORING_FLAGS_05_PATH,
]
missing_paths = [path for path in required_paths if not Path(path).exists()]
if missing_paths:
    raise FileNotFoundError("Missing required input file(s): " + ", ".join(map(str, missing_paths)))

object_db, image_db = load_nir_uco_h5(
    DB_H5_PATH,
    reconstruct_heavy_object_arrays=True,
)

guardrails_df = validate_pure_test_guardrails(
    build_pure_test_guardrails(
        train_batches=PURE_TEST_TRAIN_BATCHES,
        test_batches=PURE_TEST_BATCHES,
        train_filters=TRAIN_FILTERS,
        projection_filters=PURE_TEST_FILTERS,
        thresholds_path=VALIDATION_3WAY_SELECTED_THRESHOLDS_PATH,
        object_db=object_db,
        target_class=TARGET_CLASS,
        reference_classes=REFERENCE_CLASSES,
    )
)

if USE_WAVELENGTH_WINDOW:
    object_db, image_db, wavelengths, wavelength_info = select_wavelength_range_from_database(
        object_db=object_db,
        image_db=image_db,
        min_nm=WINDOW_MIN_NM,
        max_nm=WINDOW_MAX_NM,
    )
    wavelength_selection_df = wavelength_selection_summary(wavelength_info)
else:
    first_obj = next(iter(object_db.values()))
    wavelengths = first_obj.get("wavelengths")
    wavelengths = np.asarray(wavelengths, dtype=float) if wavelengths is not None else None
    wavelength_selection_df = pd.DataFrame()

if wavelengths is None:
    raise RuntimeError("No wavelength axis found in object_db.")

pca_selected_preprocessings_df = read_simca_table(PCA_SELECTED_PREPROCESSINGS_PATH, required=True)
preprocessing_configs_by_family = build_pca_preprocessing_configs_by_matrix_family(
    pca_selected_preprocessings_df
)
preprocessing_scope_df = pd.DataFrame([
    {
        "matrix_family": family,
        "preprocessing": name,
        "preprocessing_steps": "+".join(steps),
    }
    for family, configs in preprocessing_configs_by_family.items()
    for name, steps in configs.items()
])

candidate_panel_04c_df = read_simca_table(CANDIDATE_PANEL_04C_PATH, required=True)
track_scoring_flags_05_df = read_simca_table(TRACK_SCORING_FLAGS_05_PATH, required=True)
validation_3way_selected_thresholds_df = read_simca_table(
    VALIDATION_3WAY_SELECTED_THRESHOLDS_PATH,
    required=True,
)

wavelength_config_df = pd.DataFrame([{
    "wavelength_mode": WAVELENGTH_MODE,
    "use_wavelength_window": bool(USE_WAVELENGTH_WINDOW),
    "results_tag": RESULTS_TAG,
    "window_min_nm": WINDOW_MIN_NM if USE_WAVELENGTH_WINDOW else np.nan,
    "window_max_nm": WINDOW_MAX_NM if USE_WAVELENGTH_WINDOW else np.nan,
    "n_bands": int(len(wavelengths)),
    "min_wavelength_nm": float(np.min(wavelengths)),
    "max_wavelength_nm": float(np.max(wavelengths)),
}])

write_simca_table(wavelength_config_df, WAVELENGTH_CONFIG_PATH)
write_simca_table(preprocessing_scope_df, PREPROCESSING_SCOPE_PATH)
write_simca_table(guardrails_df, PURE_TEST_PATHS["guardrails"])

print("Number of images:", len(image_db))
print("Number of objects:", len(object_db))
print("04C candidate panel:", candidate_panel_04c_df.shape)
print("05 track scoring flags:", track_scoring_flags_05_df.shape)
print("Fixed 3-way thresholds:", validation_3way_selected_thresholds_df.shape)
display(guardrails_df)
display(wavelength_config_df)


Number of images: 48
Number of objects: 1262
04C candidate panel: (1982, 64)
05 track scoring flags: (3964, 89)
Fixed 3-way thresholds: (1982, 34)


,check_name,passed,status,details,n_records
0,train_batches_are_1_2_3,True,passed,"[1, 2, 3]",NaN
1,test_batches_are_4,True,passed,[4],NaN
2,train_test_batches_disjoint,True,passed,"{'train': [1, 2, 3], 'test': [4]}",NaN
3,projection_filters_are_pure_batch_4,True,passed,"{'sample_kind': ['pure'], 'object_nut_type': [...",NaN
4,train_filters_exclude_batch_4,True,passed,"{'sample_kind': ['pure'], 'object_nut_type': [...",NaN
5,fixed_3way_thresholds_from_04c,True,passed,C:\Users\alixg\OneDrive - Université Paris-Dau...,NaN
6,train_target_objects_available,True,passed,151 target objects,151.0
7,pure_test_reference_objects_available,True,passed,77 reference objects,77.0


,wavelength_mode,use_wavelength_window,results_tag,window_min_nm,window_max_nm,n_bands,min_wavelength_nm,max_wavelength_nm
0,non_noisy_all,False,non_noisy_all,NaN,NaN,63,960.735294,1702.0


## Build Pure-Test Candidate Panel

Notebook 05 provides reviewed `selected_config_id` values. Notebook 04C remains the source of truth for full refit parameters. Fixed 3-way thresholds must exist for every selected candidate before test evaluation starts.


In [4]:
pure_test_candidate_panel_df, pure_test_thresholds_df = select_pure_test_candidate_panel(
    candidate_panel_df=candidate_panel_04c_df,
    track_scoring_flags_df=track_scoring_flags_05_df,
    thresholds_df=validation_3way_selected_thresholds_df,
    max_candidates=MAX_PURE_TEST_CANDIDATES,
    max_candidates_per_track=MAX_PURE_TEST_CANDIDATES_PER_TRACK,
)

write_simca_table(pure_test_candidate_panel_df, PURE_TEST_PATHS["candidate_panel"])

print("Pure-test candidate panel:", pure_test_candidate_panel_df.shape)
print("Fixed 3-way thresholds for selected candidates:", pure_test_thresholds_df.shape)
display(pure_test_candidate_panel_df["matrix_family"].value_counts(dropna=False))
display(pure_test_candidate_panel_df.head())


Pure-test candidate panel: (1982, 66)
Fixed 3-way thresholds for selected candidates: (1982, 34)


matrix_family
pixel_matrix     1099
object_matrix     883
Name: count, dtype: int64

,selected_config_id,source_selected_config_id,candidate_id,model_candidate_id,refit_config_id,metric_equivalence_group_id,target_class,non_target_label,model_family,matrix_family,...,optuna_value,value_0,value_1,value_2,objective_fn_rate_max,objective_fp_rate_mean,objective_balanced_accuracy_mean,metric_equivalence_original_order,m,balanced_pixel_strategy
0,04C_refit_000008,04A_grid_000811,simca_1c2855ffea473c9e,simca_1c2855ffea473c9e,refitcfg_985add5c5b16fa12,NaN,peanut,almond,empirical_cv_rule,object_matrix,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,10,40.0,random
1,04C_refit_000009,optuna_0358,simca_9af4221efa024d1f,simca_9af4221efa024d1f,refitcfg_45a8aa0c5839ab5b,NaN,peanut,almond,rule_variant_grid,object_matrix,...,-0.712521,0.056604,0.745455,0.598971,0.056604,0.745455,0.598971,11,40.0,random
2,04C_refit_000002,optuna_0284,simca_a207c12cce354da3,simca_a207c12cce354da3,refitcfg_941887f821ed3beb,NaN,peanut,almond,rule_variant_grid,object_matrix,...,-0.534477,0.018868,0.890909,0.545111,0.018868,0.890909,0.545111,3,40.0,random
3,04C_refit_000003,optuna_0340,simca_08a074054db71001,simca_08a074054db71001,refitcfg_660f259518000307,NaN,peanut,almond,rule_variant_grid,object_matrix,...,-0.534477,0.018868,0.890909,0.545111,0.018868,0.890909,0.545111,4,40.0,random
4,04C_refit_000000,optuna_0374,simca_0d9ef35630459e28,simca_0d9ef35630459e28,refitcfg_05835c8f28287c66,metric_eq_001331,peanut,almond,rule_variant_grid,object_matrix,...,-0.445455,0.000000,0.963636,0.518182,0.000000,0.963636,0.518182,0,40.0,random


## Pure-Test Refit And Projection

When `RUN_PURE_TEST_REFIT=True`, each candidate is refit on batches 1-3 and projected onto pure batch 4. When `RUN_PURE_TEST_REFIT=False`, existing pure-test parquet outputs are loaded instead.


In [5]:
pure_test_source = "computed_refit" if RUN_PURE_TEST_REFIT else "existing_parquet"

pure_test_outputs = {
    "candidate_panel": pure_test_candidate_panel_df,
    "guardrails": guardrails_df,
}

if RUN_PURE_TEST_REFIT:
    pure_test_refit_outputs = run_pure_test_refit_batches(
                selected_configs_df=pure_test_candidate_panel_df,
                object_db=object_db,
                image_db=image_db,
                train_filters=TRAIN_FILTERS,
                projection_filters=PURE_TEST_FILTERS,
                preprocessing_configs=preprocessing_configs_by_family,
                thresholds_df=pure_test_thresholds_df,
                evaluation_stage=EVALUATION_STAGE,
                wavelengths=wavelengths,
                random_state=RANDOM_STATE,
                replace=REPLACE_BALANCED_PIXELS,
                cv_n_splits=CV_N_SPLITS,
                cv_group_col=CV_GROUP_COL,
                target_class=TARGET_CLASS,
                non_target_label=NON_TARGET_LABEL,
                batch_size=PURE_TEST_BATCH_SIZE,
                batch_metrics_dir=PURE_TEST_BATCH_METRICS_DIR,
                batch_objects_dir=PURE_TEST_BATCH_OBJECTS_DIR,
                batch_pixels_dir=PURE_TEST_BATCH_PIXELS_DIR,
                batch_3way_objects_dir=PURE_TEST_BATCH_3WAY_OBJECTS_DIR,
                save_batch_metric_tables=SAVE_BATCH_METRIC_TABLES,
                save_batch_object_tables=SAVE_BATCH_OBJECT_TABLES,
                save_batch_pixel_tables=SAVE_BATCH_PIXEL_TABLES,
                save_batch_3way_object_tables=SAVE_BATCH_3WAY_OBJECT_TABLES,
                save_combined_object_tables=SAVE_COMBINED_OBJECT_TABLES,
                save_combined_pixel_tables=SAVE_COMBINED_PIXEL_TABLES,
                save_combined_3way_object_tables=SAVE_COMBINED_3WAY_OBJECT_TABLES,
                fixed_thresholds_path=VALIDATION_3WAY_SELECTED_THRESHOLDS_PATH,
            )
    pure_test_outputs.update(pure_test_refit_outputs)
else:
    if not USE_EXISTING_PURE_TEST_OUTPUTS:
        raise RuntimeError(
            "RUN_PURE_TEST_REFIT is False and USE_EXISTING_PURE_TEST_OUTPUTS is False."
        )
    missing_existing = missing_existing_pure_test_paths(PURE_TEST_PATHS)
    if missing_existing:
        raise FileNotFoundError(
            "RUN_PURE_TEST_REFIT is False but required pure-test outputs are missing:\n"
            + "\n".join(map(str, missing_existing))
        )
    pure_test_outputs.update(load_existing_pure_test_outputs(PURE_TEST_PATHS))
    pure_test_outputs["candidate_panel"] = pure_test_candidate_panel_df
    pure_test_outputs["guardrails"] = guardrails_df
    print("Loaded existing pure-test outputs from:", RESULTS_DIR)


[pure_test_batch_4] batch_0001: candidates 1-50 / 1982
[pure_test_batch_4] 04C_refit_000008
[pure_test_batch_4] 04C_refit_000009
[pure_test_batch_4] 04C_refit_000002
[pure_test_batch_4] 04C_refit_000003
[pure_test_batch_4] 04C_refit_000000
[pure_test_batch_4] 04C_refit_000001
[pure_test_batch_4] 04C_refit_000004
[pure_test_batch_4] 04C_refit_000010
[pure_test_batch_4] 04C_refit_000005
[pure_test_batch_4] 04C_refit_000006
[pure_test_batch_4] 04C_refit_000036
[pure_test_batch_4] 04C_refit_000037
[pure_test_batch_4] 04C_refit_000038
[pure_test_batch_4] 04C_refit_000077
[pure_test_batch_4] 04C_refit_000011
[pure_test_batch_4] 04C_refit_000007
[pure_test_batch_4] 04C_refit_000012
[pure_test_batch_4] 04C_refit_000013
[pure_test_batch_4] 04C_refit_000014
[pure_test_batch_4] 04C_refit_000015
[pure_test_batch_4] 04C_refit_000016
[pure_test_batch_4] 04C_refit_000017
[pure_test_batch_4] 04C_refit_000018
[pure_test_batch_4] 04C_refit_000019
[pure_test_batch_4] 04C_refit_000020
[pure_test_batch_4] 

In [6]:
pure_test_outputs.keys()

dict_keys(['candidate_panel', 'guardrails', 'refit_metrics', '2way_object_metrics', '2way_pixel_metrics', '3way_object_metrics', 'object_image_diagnostics', 'pixel_image_diagnostics', '3way_object_image_diagnostics', 'pixel_errors_by_image', 'errors', 'objects', 'pixels', '3way_objects', 'batch_manifest'])

## Validate Pure-Test Outputs

All four SIMCA tracks are expected in a full 06A run. Missing tracks are allowed only when candidate limits are set for debugging.


In [7]:
debug_limited = MAX_PURE_TEST_CANDIDATES is not None or MAX_PURE_TEST_CANDIDATES_PER_TRACK is not None
pure_test_outputs = validate_pure_test_outputs(
    pure_test_outputs,
    expected_tracks=expcfg.SIMCA_SELECTION_TRACKS,
    allow_track_subset=debug_limited,
)

pure_test_metrics_long_df = pure_test_outputs["metrics_long"]

print("Pure-test source:", pure_test_source)
print("2-way object metrics:", pure_test_outputs["2way_object_metrics"].shape)
print("2-way pixel metrics:", pure_test_outputs["2way_pixel_metrics"].shape)
print("3-way object metrics:", pure_test_outputs["3way_object_metrics"].shape)
print("metrics long:", pure_test_metrics_long_df.shape)
print("object image diagnostics:", pure_test_outputs["object_image_diagnostics"].shape)
print("pixel image diagnostics:", pure_test_outputs["pixel_image_diagnostics"].shape)
print("3-way object image diagnostics:", pure_test_outputs["3way_object_image_diagnostics"].shape)
print("errors:", pure_test_outputs["errors"].shape)
display(pure_test_metrics_long_df.groupby(["selection_track", "metric_level"], dropna=False).size().reset_index(name="n_rows"))
display(pure_test_outputs["2way_object_metrics"].head())


Pure-test source: computed_refit
2-way object metrics: (1982, 74)
2-way pixel metrics: (1982, 49)
3-way object metrics: (1982, 64)
metrics long: (5946, 105)
object image diagnostics: (3964, 50)
pixel image diagnostics: (3964, 50)
3-way object image diagnostics: (3964, 65)
errors: (0, 0)


,selection_track,metric_level,n_rows
0,object_matrix_2way,object,883
1,object_matrix_2way,pixel,883
2,object_matrix_3way,object,883
3,pixel_matrix_2way,object,1099
4,pixel_matrix_2way,pixel,1099
5,pixel_matrix_3way,object,1099


,selected_config_id,source_selected_config_id,candidate_id,model_candidate_id,refit_config_id,metric_equivalence_group_id,target_class,non_target_label,model_family,matrix_family,...,m,balanced_pixel_strategy,non_target_class,evaluation_split,n_projected_objects,n_projected_pixels,decision_mode,evaluation_stage,metric_level,selection_track
0,04C_refit_000008,04A_grid_000811,simca_1c2855ffea473c9e,simca_1c2855ffea473c9e,refitcfg_985add5c5b16fa12,NaN,peanut,almond,empirical_cv_rule,object_matrix,...,40.0,random,almond,pure_test_batch_4,77,8401,2way,pure_test_batch_4,object,object_matrix_2way
1,04C_refit_000009,optuna_0358,simca_9af4221efa024d1f,simca_9af4221efa024d1f,refitcfg_45a8aa0c5839ab5b,NaN,peanut,almond,rule_variant_grid,object_matrix,...,40.0,random,almond,pure_test_batch_4,77,8401,2way,pure_test_batch_4,object,object_matrix_2way
2,04C_refit_000002,optuna_0284,simca_a207c12cce354da3,simca_a207c12cce354da3,refitcfg_941887f821ed3beb,NaN,peanut,almond,rule_variant_grid,object_matrix,...,40.0,random,almond,pure_test_batch_4,77,8401,2way,pure_test_batch_4,object,object_matrix_2way
3,04C_refit_000003,optuna_0340,simca_08a074054db71001,simca_08a074054db71001,refitcfg_660f259518000307,NaN,peanut,almond,rule_variant_grid,object_matrix,...,40.0,random,almond,pure_test_batch_4,77,8401,2way,pure_test_batch_4,object,object_matrix_2way
4,04C_refit_000000,optuna_0374,simca_0d9ef35630459e28,simca_0d9ef35630459e28,refitcfg_05835c8f28287c66,metric_eq_001331,peanut,almond,rule_variant_grid,object_matrix,...,40.0,random,almond,pure_test_batch_4,77,8401,2way,pure_test_batch_4,object,object_matrix_2way


## Diagnostics And Save Outputs

The diagnostics table is descriptive only. Final model selection belongs to notebook 06B.


In [8]:
pure_test_outputs["diagnostics"] = summarize_pure_test_outputs(pure_test_outputs)

protocol_settings = {
    "notebook": "06A_simca_pure_test",
    "results_tag": RESULTS_TAG,
    "wavelength_mode": WAVELENGTH_MODE,
    "input_04c_dir": RESULTS_04C_DIR,
    "input_05_dir": RESULTS_05_DIR,
    "db_h5_path": DB_H5_PATH,
    "candidate_panel_04c_path": CANDIDATE_PANEL_04C_PATH,
    "track_scoring_flags_05_path": TRACK_SCORING_FLAGS_05_PATH,
    "fixed_3way_thresholds_path": VALIDATION_3WAY_SELECTED_THRESHOLDS_PATH,
    "train_batches_json": [int(batch) for batch in PURE_TEST_TRAIN_BATCHES],
    "test_batches_json": [int(batch) for batch in PURE_TEST_BATCHES],
    "train_filters_json": TRAIN_FILTERS,
    "pure_test_filters_json": PURE_TEST_FILTERS,
    "run_pure_test_refit": bool(RUN_PURE_TEST_REFIT),
    "use_existing_pure_test_outputs": bool(USE_EXISTING_PURE_TEST_OUTPUTS),
    "pure_test_source": pure_test_source,
    "pure_test_batch_size": int(PURE_TEST_BATCH_SIZE),
    "max_pure_test_candidates": None if MAX_PURE_TEST_CANDIDATES is None else int(MAX_PURE_TEST_CANDIDATES),
    "max_pure_test_candidates_per_track": None if MAX_PURE_TEST_CANDIDATES_PER_TRACK is None else int(MAX_PURE_TEST_CANDIDATES_PER_TRACK),
    "random_state": int(RANDOM_STATE),
    "cv_n_splits": int(CV_N_SPLITS),
    "cv_group_col": CV_GROUP_COL,
    "save_batch_metric_tables": bool(SAVE_BATCH_METRIC_TABLES),
    "save_batch_object_tables": bool(SAVE_BATCH_OBJECT_TABLES),
    "save_batch_pixel_tables": bool(SAVE_BATCH_PIXEL_TABLES),
    "save_batch_3way_object_tables": bool(SAVE_BATCH_3WAY_OBJECT_TABLES),
    "save_combined_object_tables": bool(SAVE_COMBINED_OBJECT_TABLES),
    "save_combined_pixel_tables": bool(SAVE_COMBINED_PIXEL_TABLES),
    "save_combined_3way_object_tables": bool(SAVE_COMBINED_3WAY_OBJECT_TABLES),
}
pure_test_outputs["protocol"] = build_pure_test_protocol(
    protocol_settings,
    pure_test_outputs,
    final_selection_performed=False,
)

saved_paths = save_pure_test_outputs(
    pure_test_outputs,
    PURE_TEST_PATHS,
    save_combined_object_tables=SAVE_COMBINED_OBJECT_TABLES,
    save_combined_pixel_tables=SAVE_COMBINED_PIXEL_TABLES,
    save_combined_3way_object_tables=SAVE_COMBINED_3WAY_OBJECT_TABLES,
)

print("Saved:")
for path in saved_paths:
    print(" -", path)

if not SAVE_COMBINED_PIXEL_TABLES:
    print("Skipped global pixel projection table:", PURE_TEST_PATHS["pixels"])
    print("Image-level pixel diagnostics were still saved.")

display(pure_test_outputs["diagnostics"])
display(list_result_files(RESULTS_DIR).head(30))


Saved:
 - C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\06A_simca_pure_test_non_noisy_all\pure_test_candidate_panel.parquet
 - C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\06A_simca_pure_test_non_noisy_all\pure_test_2way_object_metrics.parquet
 - C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\06A_simca_pure_test_non_noisy_all\pure_test_2way_pixel_metrics.parquet
 - C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\06A_simca_pure_test_non_noisy_all\pure_test_3way_object_metrics.parquet
 - C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\06A_simca_pure_test_non_noisy_all\pure_test_metrics_long.parquet
 - C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\06A_simca_pure_test_non_noisy_all\pure_test_object_diagnostics_by_image.parquet
 - C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\06A_simca_pure_test_non_noisy_all\pure_test_pixel_diagnostics

,selection_track,matrix_family,decision_mode,metric_level,n_rows,n_candidates,best_fn_rate,best_fp_rate,best_balanced_accuracy,median_fn_rate,median_fp_rate,median_balanced_accuracy
0,object_matrix_2way,object_matrix,2way,object,883,883,0.000000,0.0000,0.767960,0.551724,0.229167,0.542744
1,object_matrix_2way,object_matrix,2way,pixel,883,883,0.006005,0.0000,0.709954,0.252212,0.585259,0.532793
2,object_matrix_3way,object_matrix,3way,object,883,883,0.000000,0.0000,1.000000,0.000000,0.041667,0.500000
3,pixel_matrix_2way,pixel_matrix,2way,object,1099,1099,0.000000,0.1875,0.868175,0.000000,0.875000,0.552083
4,pixel_matrix_2way,pixel_matrix,2way,pixel,1099,1099,0.000316,0.4367,0.735822,0.032554,0.892878,0.533346
5,pixel_matrix_3way,pixel_matrix,3way,object,1099,1099,0.000000,0.0000,1.000000,0.000000,0.437500,0.500000


,file,suffixes,size_mb
0,pure_test_metrics_long.parquet,.parquet,0.356075
1,pure_test_candidate_panel.parquet,.parquet,0.188318
2,pure_test_2way_pixel_metrics.parquet,.parquet,0.178170
3,pure_test_pixel_diagnostics_by_image.parquet,.parquet,0.172518
4,pure_test_2way_object_metrics.parquet,.parquet,0.166082
5,pure_test_3way_object_diagnostics_by_image.par...,.parquet,0.127607
6,pure_test_3way_object_metrics.parquet,.parquet,0.120196
7,pure_test_object_diagnostics_by_image.parquet,.parquet,0.101033
8,pure_test_pixel_errors_by_image.parquet,.parquet,0.080053
9,pure_test_batches\metrics\batch_0006_2way_obje...,.parquet,0.040008
